# Phase 3: 모델 학습 — ARIMA & Prophet (베이스라인)

## 이 노트북에서 하는 것
통계 기반 모델 2가지로 **베이스라인 성능**을 측정합니다.

### 베이스라인이 왜 필요한가?
LSTM 모델 RMSE가 2.5라고 할 때, 이게 좋은지 나쁜지 혼자선 모릅니다.  
가장 단순한 모델(ARIMA, Prophet)을 먼저 돌려보고,  
이후 복잡한 모델(LSTM, XGBoost)이 **얼마나 능가하는지** 비교합니다.

| 모델 | 원리 | 장점 | 단점 |
|------|------|------|------|
| ARIMA | 과거 자기 자신의 값으로 예측 | 빠름, 해석 쉬움 | 비선형 패턴 못 잡음 |
| Prophet | 추세+계절성+공휴일 분해 | 시각화 훌륭, 쉬움 | 단기 가격 예측 한계 |

In [ ]:
!pip install "pandas==2.2.2" prophet pmdarima scikit-learn -q
!apt-get install -y fonts-nanum -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
import os

from prophet import Prophet
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# ── 한글 폰트 설정 ──────────────────────────────────────────
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
# ────────────────────────────────────────────────────────────

plt.rcParams['figure.figsize'] = (14, 5)

train = pd.read_csv('/content/data/processed/train.csv', index_col=0, parse_dates=True)
val   = pd.read_csv('/content/data/processed/val.csv',   index_col=0, parse_dates=True)
test  = pd.read_csv('/content/data/processed/test.csv',  index_col=0, parse_dates=True)
print(f'학습: {len(train)} / 검증: {len(val)} / 테스트: {len(test)}')
print('한글 폰트: NanumGothic')

In [ ]:
# 평가 지표 함수
# RMSE: 예측 오차의 크기 (낮을수록 좋음)
# MAE:  평균 절대 오차
# MAPE: 가격 대비 오차 퍼센트 — 모델 간 직접 비교에 가장 유용
def evaluate(y_true, y_pred, model_name, target_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    print(f'[{model_name}] {target_name:6s} | RMSE: {rmse:.3f} | MAE: {mae:.3f} | MAPE: {mape:.2f}% | R2: {r2:.4f}')
    return {'model': model_name, 'target': target_name,
            'RMSE': round(rmse,3), 'MAE': round(mae,3), 'MAPE(%)': round(mape,2), 'R2': round(r2,4)}

all_results = []

In [ ]:
# ===========================================
# ARIMA 모델
# auto_arima: 최적 p,d,q 파라미터 자동 탐색
# p = 자기회귀 차수 / d = 차분 / q = 이동평균 차수
# ===========================================
print('ARIMA 파라미터 탐색 중... (약 1~2분)')
train_close = train['Close'].values

arima_model = auto_arima(
    train_close,
    max_p=5, max_q=5, d=None,
    seasonal=False,
    information_criterion='aic',
    trace=True,
    suppress_warnings=True,
    stepwise=True
)
print(f'\n최적 ARIMA: {arima_model.order}')

In [ ]:
# ARIMA 테스트셋 예측 (Walk-forward Validation)
# 매일 한 스텝씩 예측 → 실제값 반영 후 다음 예측 = 실전과 동일 조건
print('Walk-forward 예측 중...')
history = list(train_close) + list(val['Close'].values)
test_close = test['Close'].values
arima_preds = []

for i in range(len(test_close)):
    m = auto_arima(history, order=arima_model.order, seasonal=False,
                   suppress_warnings=True, error_action='ignore')
    arima_preds.append(m.predict(n_periods=1)[0])
    history.append(test_close[i])
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(test_close)}')

arima_preds = np.array(arima_preds)
test_atr = test['atr_14'].values

# ARIMA는 종가만 예측 → ATR 이용해 고/저가 추정
# ATR(Average True Range): 하루 평균 가격 변동 범위
# 예상 고가 = 예측 종가 + ATR*0.5 / 예상 저가 = 예측 종가 - ATR*0.5
arima_high = arima_preds + test_atr * 0.5
arima_low  = arima_preds - test_atr * 0.5

print('\n--- ARIMA 성능 ---')
all_results.append(evaluate(test['target_close'].values, arima_preds, 'ARIMA', 'Close'))
all_results.append(evaluate(test['target_high'].values,  arima_high,  'ARIMA', 'High'))
all_results.append(evaluate(test['target_low'].values,   arima_low,   'ARIMA', 'Low'))

In [ ]:
# ARIMA 예측 시각화
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test['Close'], label='실제 종가', color='white', lw=1.5)
ax.plot(test.index, arima_preds,  label='ARIMA 예측', color='#ffd700', lw=1.5, ls='--')
ax.fill_between(test.index, arima_low, arima_high, alpha=0.2, color='#ffd700', label='예측 범위 (±0.5 ATR)')
ax.set_title('ARIMA 예측 결과 (테스트셋)')
ax.set_ylabel('가격 (USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/arima_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================
# Prophet 모델
# Meta(Facebook)의 시계열 예측 라이브러리
# 추세 + 주간계절성 + 연간계절성 자동 분해
# ===========================================
train_val = pd.concat([train, val])
prophet_df = pd.DataFrame({'ds': train_val.index, 'y': train_val['Close'].values})

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,  # 추세 변화 민감도
    interval_width=0.80            # 80% 신뢰구간
)
prophet_model.fit(prophet_df)
print('Prophet 학습 완료!')

future_df = pd.DataFrame({'ds': test.index})
forecast  = prophet_model.predict(future_df)

print('\n--- Prophet 성능 ---')
all_results.append(evaluate(test['target_close'].values, forecast['yhat'].values, 'Prophet', 'Close'))
all_results.append(evaluate(test['target_high'].values,  forecast['yhat_upper'].values, 'Prophet', 'High'))
all_results.append(evaluate(test['target_low'].values,   forecast['yhat_lower'].values, 'Prophet', 'Low'))

In [ ]:
# Prophet 시각화
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test['Close'], label='실제 종가', color='white', lw=1.5)
ax.plot(forecast['ds'], forecast['yhat'], label='Prophet 예측', color='#26a69a', lw=1.5, ls='--')
ax.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                alpha=0.2, color='#26a69a', label='80% 신뢰구간')
ax.set_title('Prophet 예측 결과 (테스트셋)')
ax.set_ylabel('가격 (USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/prophet_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

# Prophet 컴포넌트 분해 — 발표에서 아주 유용
fig2 = prophet_model.plot_components(forecast)
plt.savefig('/content/prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 결과 저장
os.makedirs('/content/results', exist_ok=True)
pd.DataFrame(all_results).to_csv('/content/results/baseline_results.csv', index=False)

for name, preds, high, low in [
    ('arima', arima_preds, arima_high, arima_low),
    ('prophet', forecast['yhat'].values, forecast['yhat_upper'].values, forecast['yhat_lower'].values)
]:
    pd.DataFrame({
        'date': test.index, 'pred_close': preds, 'pred_high': high, 'pred_low': low,
        'actual_high': test['target_high'].values,
        'actual_low': test['target_low'].values,
        'actual_close': test['target_close'].values
    }).to_csv(f'/content/results/{name}_predictions.csv', index=False)

print('베이스라인 결과 저장 완료!')
print('\n다음: 04_models_xgboost_lstm.ipynb')